# Natural-Language Video Search with TwelveLabs + FiftyOne

The **TwelveLabs integration ships natively in FiftyOne 1.19+** — no patched
modules or workarounds. This notebook is a self-contained demo you can run on any
folder of videos:

- **Marengo** → 512-d, text-aligned video embeddings for natural-language search
- **Pegasus** → zero-shot captioning + promptable Q&A
- FiftyOne Brain → similarity, uniqueness, and an interactive embeddings map

Everything runs **server-side via the TwelveLabs API** — no GPU required.

### What you need
1. `pip install fiftyone twelvelabs` (done in the first cell).
2. A **TwelveLabs API key** — free at https://playground.twelvelabs.io/dashboard/api-key
3. `ffmpeg`/`ffprobe` on your PATH (for the optional duration filter).
4. A folder of video clips. This demo uses a small public sample set by default,
   but you can point it at your own videos with one variable.

> Run the cells top to bottom. If a cell installs packages, restart the kernel when
> prompted, then continue.

## 1. Install

In [ ]:
import sys
!{sys.executable} -m pip install -q -U "fiftyone>=1.19" "twelvelabs>=1.2.8"
print("installed — if this is a fresh install, restart the kernel before continuing")

## 2. Verify the integration is available

Confirms FiftyOne is 1.19+ and the bundled TwelveLabs model loads. If the import
path has changed in your version, check the current docs at
https://docs.voxel51.com/integrations/ and adjust the import below.

In [ ]:
import fiftyone as fo
print("fiftyone:", fo.__version__)

from fiftyone.utils.twelvelabs import TwelveLabsModel, TwelveLabsModelConfig
import twelvelabs
print("twelvelabs SDK:", twelvelabs.__file__)
assert hasattr(twelvelabs, "TwelveLabs"), (
    "'twelvelabs' is shadowed by a local file of the same name — "
    "rename any twelvelabs.py in your working directory and restart the kernel"
)
print("TwelveLabs integration ready")

## 3. Set your TwelveLabs API key

In [ ]:
import os, getpass
if not os.environ.get("TWELVELABS_API_KEY"):
    os.environ["TWELVELABS_API_KEY"] = getpass.getpass("Enter your TWELVELABS_API_KEY: ")
print("API key set:", bool(os.environ.get("TWELVELABS_API_KEY")))

## 4. Choose your videos

By default this downloads a handful of small public sample clips so the notebook is
runnable out of the box. **To use your own footage**, set `USE_SAMPLE_VIDEOS = False`
and point `DATA_DIR` at a folder of `.mp4` files.

TwelveLabs' Marengo has ingest requirements (roughly: resolution ≥ 360×360, a
standard aspect ratio, and duration ≥ 4 seconds). The cell after this filters out
clips that are too short.

In [ ]:
import os

USE_SAMPLE_VIDEOS = True
DATA_DIR          = os.path.expanduser("~/my_videos")   # used only if USE_SAMPLE_VIDEOS = False
NUM_CLIPS         = 12                                   # how many clips to embed
MIN_DURATION      = 4.0                                  # seconds; Marengo needs >= 4s
DATASET_NAME      = "twelvelabs-demo"

if USE_SAMPLE_VIDEOS:
    # Long-stable public sample MP4s (H.264, 16:9, several seconds each).
    import urllib.request
    BUCKET = "https://storage.googleapis.com/gtv-videos-bucket/sample"
    SAMPLES = [
        "BigBuckBunny.mp4", "ElephantsDream.mp4", "ForBiggerBlazes.mp4",
        "ForBiggerEscapes.mp4", "ForBiggerFun.mp4", "ForBiggerJoyrides.mp4",
        "ForBiggerMeltdowns.mp4", "Sintel.mp4", "SubaruOutbackOnStreetAndDirt.mp4",
        "TearsOfSteel.mp4", "VolkswagenGTIReview.mp4", "WeAreGoingOnBubbles.mp4",
    ]
    DATA_DIR = os.path.expanduser("~/twelvelabs_sample_videos")
    os.makedirs(DATA_DIR, exist_ok=True)
    for fn in SAMPLES:
        dst = os.path.join(DATA_DIR, fn)
        if not os.path.exists(dst):
            print("downloading", fn)
            urllib.request.urlretrieve(f"{BUCKET}/{fn}", dst)
    print("sample videos ready in", DATA_DIR)

assert os.path.isdir(DATA_DIR), f"video folder not found: {DATA_DIR}"
print("data dir:", DATA_DIR)

## 5. Select clips that meet the duration requirement

`ffprobe` each file and keep those ≥ `MIN_DURATION` until we have `NUM_CLIPS`.

In [ ]:
import glob, json, subprocess, shutil

assert shutil.which("ffprobe"), "ffprobe not found — install ffmpeg"

def duration(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=duration", "-of", "json", path],
        capture_output=True, text=True,
    )
    try:
        d = json.loads(out.stdout)["streams"][0].get("duration")
        return float(d) if d else None
    except Exception:
        return None

all_clips = sorted(glob.glob(os.path.join(DATA_DIR, "*.mp4")))
eligible = []
for p in all_clips:
    d = duration(p)
    if d is not None and d >= MIN_DURATION:
        eligible.append(p)
    if len(eligible) >= NUM_CLIPS:
        break

print(f"{len(eligible)} eligible clips (of {len(all_clips)} scanned)")
assert eligible, "no eligible clips — lower MIN_DURATION or check DATA_DIR"

## 6. Build the FiftyOne dataset

In [ ]:
if fo.dataset_exists(DATASET_NAME):
    fo.delete_dataset(DATASET_NAME)

dataset = fo.Dataset(DATASET_NAME, persistent=True)
dataset.add_samples([fo.Sample(filepath=p) for p in eligible])
print("samples:", len(dataset))

## 7. Smoke-test one embedding

Confirms the API works end to end before embedding everything. Expect `(512,)`.
Each embedding is a live API call, so a single clip may take ~30–60 s.

In [ ]:
embed_model = TwelveLabsModel(TwelveLabsModelConfig({"operation": "embed"}))
v = embed_model.embed(dataset.first().filepath)
print("embedding shape:", v.shape)

## 8. Launch the FiftyOne App

Opens in a browser tab. Keep this `session` around — later cells update
`session.view`, and you refresh the tab to see results.

In [ ]:
session = fo.launch_app(dataset, auto=False)
session.open_tab()
print("App URL:", session.url)

## 9. Embed every clip with Marengo

`skip_failures=False` surfaces any error instead of leaving silent `None` fields.

In [ ]:
dataset.compute_embeddings(
    embed_model, embeddings_field="twelvelabs", skip_failures=False
)
print("embedded:", dataset.exists("twelvelabs").count(), "/", len(dataset))
print("shape:  ", dataset.first()["twelvelabs"].shape)

## 10. Build a similarity index

In [ ]:
import fiftyone.brain as fob

if "tl_sim" in dataset.list_brain_runs():
    dataset.delete_brain_run("tl_sim")

fob.compute_similarity(
    dataset, model=embed_model, embeddings="twelvelabs", brain_key="tl_sim"
)
print("index 'tl_sim' ready")

## 11. 🔎 Search by natural-language description

Type a description; clips rank by semantic relevance. We save each query as a
**saved view** so it appears in the App's view dropdown — refresh the tab and pick it.

Edit `QUERIES` to match your footage.

In [ ]:
QUERIES = {
    "car on a dirt road":     "a car driving on a dirt road",
    "animated characters":    "an animated cartoon character",
    "person indoors":         "a person sitting indoors",
}

for name, query in QUERIES.items():
    view = dataset.sort_by_similarity(query, k=NUM_CLIPS, brain_key="tl_sim")
    if name in dataset.list_saved_views():
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)

print("saved views:", dataset.list_saved_views())
print("\nRefresh the App tab and pick a view from the dropdown.")

Prefer the notebook? Point the live App at one of the views directly:

In [ ]:
session.view = dataset.load_saved_view(list(QUERIES.keys())[0])
print("App now showing:", list(QUERIES.keys())[0])

## 12. Auto-caption with Pegasus

Zero-shot natural-language description per clip, written to a `caption` field.
Toggle the `caption` field in the App sidebar to see them on each clip.

In [ ]:
caption_model = TwelveLabsModel(TwelveLabsModelConfig({"operation": "caption"}))
dataset.apply_model(caption_model, label_field="caption")

for s in dataset.take(5):
    print("-", s.caption.label)

## 13. Promptable Q&A

Same model, custom `prompt` — ask anything and store the answer as a label.

In [ ]:
qa_model = TwelveLabsModel(
    TwelveLabsModelConfig({
        "operation": "caption",
        "prompt": "In 3 words or fewer, what is the main subject of this video?",
        "max_tokens": 512,
    })
)
dataset.apply_model(qa_model, label_field="subject")

for s in dataset.take(5):
    print("-", s.subject.label)

## 14. Curate with FiftyOne Brain

Use the embeddings to find near-duplicates and rank clips by uniqueness — the
foundation of building a diverse, balanced dataset.

In [ ]:
# uniqueness: adds a 0–1 score per clip (sort/filter it in the App sidebar)
fob.compute_uniqueness(dataset)

# near-duplicates from the similarity index
results = dataset.load_brain_results("tl_sim")
results.find_duplicates(thresh=0.25)          # lower = stricter
dups = results.duplicates_view()
if "near-duplicates" in dataset.list_saved_views():
    dataset.delete_saved_view("near-duplicates")
dataset.save_view("near-duplicates", dups)

if "most-unique" in dataset.list_saved_views():
    dataset.delete_saved_view("most-unique")
dataset.save_view("most-unique", dataset.sort_by("uniqueness", reverse=True))

print("duplicate clips:", len(dups))
print("saved views:", dataset.list_saved_views())

## 15. Visualize the embedding space

Project the 512-d embeddings to 2D (PCA, no extra deps). Open the **Embeddings**
panel in the App to see clips cluster by content.

In [ ]:
if "tl_viz" in dataset.list_brain_runs():
    dataset.delete_brain_run("tl_viz")

fob.compute_visualization(
    dataset, embeddings="twelvelabs", brain_key="tl_viz", method="pca"
)
session.view = dataset.view()
print("visualization ready — open the Embeddings panel in the App")

## Recap

With FiftyOne 1.19+ the TwelveLabs integration is built in — `pip install`, set an
API key, and go. In this notebook you:

- Embedded video with **Marengo** (512-d, server-side, no GPU)
- Searched footage in plain English and saved the queries as App views
- Auto-captioned and ran promptable Q&A with **Pegasus**
- Curated with uniqueness + near-duplicate detection
- Visualized the embedding space

**Use your own data:** set `USE_SAMPLE_VIDEOS = False` and `DATA_DIR` in step 4.
**Scale up:** raise `NUM_CLIPS` (mind your API minutes).

### Resources
- FiftyOne integrations & docs: https://docs.voxel51.com/integrations/
- FiftyOne Brain (similarity, uniqueness, visualization): https://docs.voxel51.com/user_guide/brain.html
- TwelveLabs docs: https://docs.twelvelabs.io
- TwelveLabs Playground / API keys: https://playground.twelvelabs.io